# Transfer Squad Selection Optimizer


## Importing Libraries
Import necessary libraries

In [128]:
import pandas as pd
import os
import numpy as np
import pulp
from pulp import LpProblem, LpMaximize, LpVariable, lpSum, LpBinary, LpStatus
from tabulate import tabulate 
import requests
import io
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

## Getting the Data
Functions to read and clean up the data

In [129]:
class get_dataset():
    def read_file(self, season):
        base_url = f'https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/{season}/gws/merged_gw.csv'
        
        retry_strategy = Retry(
            total=5,  
            status_forcelist=[429, 500, 502, 503, 504], 
            allowed_methods=["HEAD", "GET", "OPTIONS"],  
            backoff_factor=1 
        )
        adapter = HTTPAdapter(max_retries=retry_strategy)
        http = requests.Session()
        http.mount("https://", adapter)
        http.mount("http://", adapter)
        try:
            response = http.get(base_url)
            response.raise_for_status()  
            df = pd.read_csv(io.StringIO(response.text))
            return df
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            return None
    
    def make_columns(self, df):
        df['player_id'] = df.groupby(['name', 'team']).ngroup()
        df['merged'] = False
        df['filled'] = False
        df = df.replace({'position': {'GKP': 'GK'}})
        columns = ['name', 'position', 'xP', 'value', 'GW', 'total_points', 'team', 'filled', 'merged', 'player_id','minutes']
        df = df[columns]
        return df

    def fill_squads(self, df):
        unique_players_df = df[['player_id', 'name', 'position', 'team']].drop_duplicates()
        unique_gws_df = df['GW'].unique()

        player_avg_values = df.groupby('player_id')['value'].mean().reset_index()

        complete_index = pd.MultiIndex.from_product([unique_players_df['player_id'], unique_gws_df], names=['player_id', 'GW'])
        complete_df = pd.DataFrame(index=complete_index).reset_index()

        complete_df = complete_df.merge(unique_players_df, on='player_id', how='left')
        df = complete_df.merge(df, on=['player_id', 'GW', 'name', 'position', 'team'], how='left')
        df = df.merge(player_avg_values, on='player_id', how='left', suffixes=('', '_avg'))

        df['filled'] = df['value'].isnull()
        df.loc[df['value'].isnull(), 'value'] = df['value_avg'] 

        df.drop(columns=['value_avg'], inplace=True)
        df['team_encoded'] = df['team'].astype('category').cat.codes + 1
        df = pd.concat([df, pd.get_dummies(df['position'], prefix='', prefix_sep='')], axis=1)
        return df
    
    def merge_rows(self, df):
        df['merged'] = df.groupby(['player_id', 'GW'])['player_id'].transform('size').gt(1) 
        aggregation_functions = {
            'name': 'first',
            'position': 'first',
            'team_encoded': 'first',
            'xP': 'sum',
            'value': 'first',
            'total_points': 'sum',
            'team': 'first',
            'filled': 'first',
            'merged': 'first',
            'GK': 'first',
            'DEF': 'first',
            'MID': 'first',
            'FWD': 'first',
            'minutes': 'first'
        }
        df = df.groupby(['player_id', 'GW']).agg(aggregation_functions).reset_index()
        return df
    
    def round_df(self, df):
        df['value'] = df['value'].round(0).astype(int)
        df['total_points'] = df['total_points'].round(0).astype(int)
        df['xP'] = df['xP'].round(1)
        return df

    
    def run(self, season):
        df = self.read_file(season)
        df = self.make_columns(df)
        df = self.fill_squads(df)
        df = self.merge_rows(df)
        df = self.round_df(df)
        df = df.infer_objects()
        return df



## Functions ##
Define various functions to improve code readability

In [130]:
class functions:
    def get_previous_season_data(self, season):
        previous_season = f"{int(season[:4]) - 1}-{int(season[5:]) - 1:02d}"
        return get_dataset().run(previous_season)

    def get_bench_selection(self, df, season):
        df_old = self.get_previous_season_data(season)
        avg_minutes_per_game = df_old.groupby('player_id')['minutes'].sum() / 38
        valid_player_ids = avg_minutes_per_game.loc[lambda x: x >= 60].index
        valid_names = df_old[df_old['player_id'].isin(valid_player_ids)]['name'].unique()
        bench_selection_df_filtered = pd.DataFrame(set(df['name']).intersection(valid_names), columns=['name'])
        
        return bench_selection_df_filtered
    def assign_captaincies(self, selected_squad, starters):
        selected_squad['role'] = ['starting' if i in starters else 'bench' for i in selected_squad.index]
        highest_xP_player_index = selected_squad[selected_squad['role'] == 'starting']['xP'].idxmax()
        selected_squad.loc[:, 'captain'] = 0 
        selected_squad.at[highest_xP_player_index, 'captain'] = 1 

        remaining_players_df = selected_squad.drop(highest_xP_player_index)
        second_highest_xP_player_index = remaining_players_df[remaining_players_df['role'] == 'starting']['xP'].idxmax()
        selected_squad.loc[:, 'vice_captain'] = 0 
        selected_squad.at[second_highest_xP_player_index, 'vice_captain'] = 1
        selected_squad.update(selected_squad[['captain', 'vice_captain']])
        return selected_squad
        
    def violates_positional_constraints(self, starting_lineup, potential_sub):
        temp_starting = starting_lineup.copy()
        temp_starting.loc[potential_sub.name] = potential_sub
        defenders = temp_starting[temp_starting['position'] == 'DEF'].shape[0]
        midfielders = temp_starting[temp_starting['position'] == 'MID'].shape[0]
        forwards = temp_starting[temp_starting['position'] == 'FWD'].shape[0]
        return not (3 <= defenders <= 5 and 3 <= midfielders <= 5 and 1 <= forwards <= 3) 
    
    def prepare_squad_score(self, full_squad):
        starting_lineup = full_squad[full_squad['role'] == 'starting']
        bench = full_squad[full_squad['role'] == 'bench']
        non_playing_starters = starting_lineup[(starting_lineup['minutes'] == 0)]

        for index, player in non_playing_starters.iterrows():
            if player['position'] == 'GK':  #
                bench_gk = bench[(bench['position'] == 'GK') & ((bench['minutes'] > 0))]  
                if not bench_gk.empty:  
                    bench_gk_index = bench_gk.index[0]  
                    starting_lineup.loc[index], bench.loc[bench_gk_index] = bench.loc[bench_gk_index].copy(), starting_lineup.loc[index].copy()
                    print(f"Swapped {starting_lineup.loc[index, 'name']} with {bench.loc[bench_gk_index, 'name']}")
            else: 
                valid_bench_players = []
                for bench_index, bench_player in bench.iterrows():
                    if bench_player['position'] != 'GK' and (bench_player['minutes'] > 0):
                        if not self.violates_positional_constraints(starting_lineup, bench_player):
                            valid_bench_players.append((bench_index, bench_player))
                if valid_bench_players:
                    highest_scoring_player = max(valid_bench_players, key=lambda x: x[1]['total_points'])
                    highest_scoring_index = highest_scoring_player[0]
                    starting_lineup.loc[index], bench.loc[highest_scoring_index] = bench.loc[highest_scoring_index].copy(), starting_lineup.loc[index].copy()
                    print(f"Swapped {starting_lineup.loc[index, 'name']} with {bench.loc[highest_scoring_index, 'name']}")
        non_playing_starters = starting_lineup[(starting_lineup['minutes'] == 0)]
        return starting_lineup
    
    def calculate_squad_score(self, full_squad):
        starting_lineup = self.prepare_squad_score(full_squad)
        captain_name = None
        vice_captain_name = None
        flag = False

        vice_captain_row = full_squad[full_squad['vice_captain'] == 1]
        if not vice_captain_row.empty:
            vice_captain_name = vice_captain_row.iloc[0]['name']
        captain_row = full_squad[full_squad['captain'] == 1]
        if not captain_row.empty:
            captain_name = captain_row.iloc[0]['name']
            if not captain_row.iloc[0]['minutes'] == 0:
                flag = False
                captain_points = captain_row.iloc[0]['total_points'] * 2  
            else:
                flag = True
                captain_points = vice_captain_row.iloc[0]['total_points'] *2 
        else:
            captain_points = 0       
        if flag == False:
            other_players_points = starting_lineup[starting_lineup['captain'] != 1]['total_points'].sum()
        else:
            other_players_points = starting_lineup[starting_lineup['vice_captain'] != 1]['total_points'].sum()
        total_points_with_captain_doubled = captain_points + other_players_points
        return total_points_with_captain_doubled, captain_name, vice_captain_name
    
    def sort_squad(self, squad):
        positions = ['GK', 'DEF', 'MID', 'FWD']
        column_order = ['name', 'position', 'team', 'role', 'minutes', 'xP', 'total_points', 'value', 'captain', 'vice_captain', 'player_id']
        squad = squad.reindex(columns=column_order)
        return pd.concat([
            squad[squad['role'] == 'starting'].sort_values(
                by='position', key=lambda x: x.map({p: i for i, p in enumerate(positions)})
            ),
            squad[squad['role'] == 'bench'].sort_values(  
                by='position', key=lambda x: x.map({p: i for i, p in enumerate(positions)})
            )
        ])
    
    def data_calculations(self, full_squad, budget_limit, total_score, paid_transfers):
        total_budget_used = full_squad['value'].sum()
        positions_count = full_squad[full_squad['role'] == 'starting'][['GK', 'DEF', 'MID', 'FWD']].sum()
        positions_count_str = ",".join(str(count) for count in positions_count)
        total_points, captain_name, vice_captain_name = self.calculate_squad_score(full_squad)
        total_points = total_points - (paid_transfers) * 4
        total_score += total_points 
        total_budget_used = full_squad['value'].sum()
        data = [
            ["Total Budget Used", total_budget_used],
            ["Budget Limit", budget_limit],
            ["Positions Count", positions_count_str],
            ["Captain", captain_name],
            ["Vice-Captain", vice_captain_name],
            ["Paid Transfers", (paid_transfers)],
            ["Total Points", total_points],
            ["Total Season Points", total_score]
        ]
        return total_score, data
    
    def update_data(self, data, new_total_points):
        for item in data:
            if item[0] == "Total Points":
                item[1] += new_total_points
                for sub_item in data:
                    if sub_item[0] == "Total Season Points":
                        sub_item[1] += new_total_points
                        break
                break
        return new_total_points

## WildCards Class ##

In [131]:
class WildCards():
    def get_threshold(self, average_quantity):
        historical_avg = pd.Series(average_quantity).mean()
        historical_std = pd.Series(average_quantity).std()
        threshold = historical_avg + historical_std
        return threshold

    def bench_boost_calc(self, full_squad, average_xp_bench):
        bench_xp = full_squad[full_squad['role'] == 'bench']['xP'].mean()
        average_xp_bench.append(bench_xp)
        bench_boost_threshold = self.get_threshold(average_xp_bench)

        return bench_boost_threshold, bench_xp, average_xp_bench
    
    def bench_boost(self, full_squad, average_xp_bench, bench_boost_activated):
        bench_boost_threshold, average_xp, average_xp_bench = self.bench_boost_calc(full_squad, average_xp_bench)
        if not bench_boost_activated and average_xp > bench_boost_threshold:
            print("Bench Boost Activated!")
            return True
        return False
    
    def triple_captain_calc(self, full_squad, average_xp_captain):
        captain_xP = full_squad[full_squad['captain'] == 1]['xP'].sum()
        captain_name = full_squad[full_squad['captain'] == 1]['name'].iloc[0]
        average_xp_captain.append(captain_xP)
        triple_captain_threshold = self.get_threshold(average_xp_captain)
        return triple_captain_threshold, captain_xP, average_xp_captain, captain_name
    
    def triple_captain(self, full_squad, average_xp_captain, triple_captain_activated):
        triple_captain_threshold, captain_xP, average_xp_captain, captain_name = self.triple_captain_calc(full_squad, average_xp_captain)
        if not triple_captain_activated and captain_xP > triple_captain_threshold:
            print(f"Triple Captain Activated! {captain_name} will get triple points.")
            return True
        return False
    
    def wildcard_calc(self, paid_transfers, average_paid_transfers):
        average_paid_transfers.append(paid_transfers)
        wildcard_threshold = self.get_threshold(average_paid_transfers)

        return wildcard_threshold, paid_transfers, average_paid_transfers
    
    def wildcard(self, paid_transfers, average_paid_transfers, wildcard_activated):
        wildcard_threshold, paid_transfers, average_paid_transfers = self.wildcard_calc(paid_transfers, average_paid_transfers)
        if not wildcard_activated and paid_transfers > wildcard_threshold:
            print(f"Wildcard Activated! {paid_transfers -1} paid transfers refunded.")
            return True
        return False
    
    def freehit_calc(self, full_squad, average_xp_squad):
        squad_xp = full_squad[full_squad['role'] == 'starting']['xP'].sum()
        average_xp_squad.append(squad_xp)
        freehit_threshold = self.get_threshold(average_xp_squad)
        return freehit_threshold, squad_xp, average_xp_squad
    
    def freehit(self, full_squad, average_xp_squad, free_hit_activated):
        freehit_threshold, squad_xp, average_xp_squad = self.freehit_calc(full_squad, average_xp_squad)
        if not free_hit_activated and squad_xp < freehit_threshold:
            print("Free Hit Activated!")
            return True
        return False



## Constraints ##
Adding constraints for the squad selection

In [132]:
class constraints:
    def get_decision_variables(self, df):
        decision_variables = {
        "transfer_in_starting_free": LpVariable.dicts("transfer_in_starting_free", df.index, 0, 1, LpBinary),
        "transfer_in_starting_paid": LpVariable.dicts("transfer_in_starting_paid", df.index, 0, 1, LpBinary),
        "transfer_out_starting_free": LpVariable.dicts("transfer_out_starting_free", df.index, 0, 1, LpBinary),
        "transfer_out_starting_paid": LpVariable.dicts("transfer_out_starting_paid", df.index, 0, 1, LpBinary),
        "stay_starting": LpVariable.dicts("stay_starting", df.index, 0, 1, LpBinary),
        "transfer_in_bench_free": LpVariable.dicts("transfer_in_bench_free", df.index, 0, 1, LpBinary),
        "transfer_in_bench_paid": LpVariable.dicts("transfer_in_bench_paid", df.index, 0, 1, LpBinary),
        "transfer_out_bench_free": LpVariable.dicts("transfer_out_bench_free", df.index, 0, 1, LpBinary),
        "transfer_out_bench_paid": LpVariable.dicts("transfer_out_bench_paid", df.index, 0, 1, LpBinary),
        "stay_bench": LpVariable.dicts("stay_bench", df.index, 0, 1, LpBinary),
        "bench_to_starting": LpVariable.dicts("bench_to_starting", df.index, 0, 1, LpBinary),
        "starting_to_bench": LpVariable.dicts("starting_to_bench", df.index, 0, 1, LpBinary)
    }
        return decision_variables
    
    def set_objective_function(self, df, decision_variables, prob):
        # Objective function: Maximize xP from starting players only
        prob += lpSum(decision_variables["transfer_in_starting_free"][i] * df.loc[i, 'xP'] +
                      decision_variables["transfer_in_starting_paid"][i] * (df.loc[i, 'xP'] - 4) +
                      decision_variables["stay_starting"][i] * df.loc[i, 'xP'] +
                      decision_variables["bench_to_starting"][i] * df.loc[i, 'xP'] -
                      decision_variables["transfer_in_bench_paid"][i] * 4 for i in df.index), "TotalExpectedPoints"
    
    def apply_formation_constraints(self, df, decision_variables, prob):
        # Squad composition constraints
        prob += lpSum(decision_variables["transfer_in_starting_free"][i] + 
                    decision_variables["transfer_in_starting_paid"][i] + 
                    decision_variables["stay_starting"][i] + 
                    decision_variables["bench_to_starting"][i] for i in df.index) == 11, "11Starters"        
        prob += lpSum(decision_variables["transfer_in_bench_free"][i] +
                    decision_variables["transfer_in_bench_paid"][i] +
                    decision_variables["stay_bench"][i] +
                    decision_variables["starting_to_bench"][i] for i in df.index) == 4, "4Bench"
        

        positions = ['GK', 'DEF', 'MID', 'FWD']
        min_starting = [1, 3, 3, 1]
        max_starting = [1, 5, 5, 3]
        total = [2,5,5,3]

        for pos, min_start, max_start, tot in zip(positions, min_starting, max_starting, total):
            prob += lpSum(decision_variables["transfer_in_starting_free"][i] + 
                    decision_variables["transfer_in_starting_paid"][i] + 
                    decision_variables["stay_starting"][i] + 
                    decision_variables["bench_to_starting"][i] for i in df[df['position'] == pos].index) >= min_start, f"Min{min_start}Starting{pos}"
            
            prob += lpSum(decision_variables["transfer_in_starting_free"][i] + 
                    decision_variables["transfer_in_starting_paid"][i] + 
                    decision_variables["stay_starting"][i] + 
                    decision_variables["bench_to_starting"][i] for i in df[df['position'] == pos].index) <= max_start, f"Max{max_start}Starting{pos}"
            
            prob += lpSum(decision_variables["transfer_in_starting_free"][i] + 
                    decision_variables["transfer_in_starting_paid"][i] + 
                    decision_variables["stay_starting"][i] + 
                    decision_variables["bench_to_starting"][i] + 
                    decision_variables["transfer_in_bench_free"][i] +
                    decision_variables["transfer_in_bench_paid"][i] +
                    decision_variables["stay_bench"][i] +
                    decision_variables["starting_to_bench"][i] for i in df[df['position'] == pos].index) == tot, f"Total{tot}{pos}"

    def apply_transfer_constraints(self, df, decision_variables, prob, budget_limit):            
        prob += lpSum(
            decision_variables["transfer_in_starting_free"][i] * df.loc[i, 'value'] + 
            decision_variables["transfer_in_starting_paid"][i] * df.loc[i, 'value'] + 
            decision_variables["transfer_in_bench_free"][i] * df.loc[i, 'value'] +
            decision_variables["transfer_in_bench_paid"][i] * df.loc[i, 'value'] +
            decision_variables["stay_starting"][i] * df.loc[i, 'value'] + 
            decision_variables["stay_bench"][i] * df.loc[i, 'value'] +
            decision_variables["bench_to_starting"][i] * df.loc[i, 'value'] + 
            decision_variables["starting_to_bench"][i] * df.loc[i, 'value'] for i in df.index) <= budget_limit, "Budget"

        for team in df['team_encoded'].unique():
            prob += lpSum(decision_variables["transfer_in_starting_free"][i] + 
                      decision_variables["transfer_in_starting_paid"][i] + 
                      decision_variables["stay_starting"][i] + 
                      decision_variables["bench_to_starting"][i] + 
                      decision_variables["transfer_in_bench_free"][i] +
                      decision_variables["transfer_in_bench_paid"][i] +
                      decision_variables["stay_bench"][i] +
                      decision_variables["starting_to_bench"][i] for i in df[df['team_encoded'] == team].index) <= 3, f"Max3FromTeam_{team}"
            
    def apply_player_constraints(self, df, decision_variables, prob, previous_squad, gw, transfer_limit, season):
        if gw == 1:
            prob += lpSum(decision_variables["transfer_in_starting_paid"][i] for i in df.index) == 0, f"StartingPaidTransfersLimit"
            prob += lpSum(decision_variables["transfer_in_bench_paid"][i] for i in df.index) == 0, f"BenchPaidTransfersLimit"
            prob += lpSum(decision_variables["bench_to_starting"][i] for i in df.index) == 0, f"BenchToStartingTransfersLimit"
            prob += lpSum(decision_variables["starting_to_bench"][i] for i in df.index) == 0, f"StartingToBenchTransfersLimit"
            prob += lpSum(decision_variables["stay_starting"][i] for i in df.index) == 0, f"StayStartingTransfersLimit"
            prob += lpSum(decision_variables["stay_bench"][i] for i in df.index) == 0, f"StayBenchTransfersLimit"
            prob += lpSum(decision_variables["transfer_out_starting_free"][i] for i in df.index) == 0, f"TransferOutStartingFreeLimit"
            prob += lpSum(decision_variables["transfer_out_starting_paid"][i] for i in df.index) == 0, f"TransferOutStartingPaidLimit"
            prob += lpSum(decision_variables["transfer_out_bench_free"][i] for i in df.index) == 0, f"TransferOutBenchFreeLimit"
            prob += lpSum(decision_variables["transfer_out_bench_paid"][i] for i in df.index) == 0, f"TransferOutBenchPaidLimit"
            if not season == '2020-21':
                bench_selection = functions().get_bench_selection(df, season)
                for i in df.index:
                    if df.loc[i, 'name'] not in bench_selection['name'].values:
                        prob += decision_variables["transfer_in_bench_paid"][i] == 0, f"NonBenchPaid{i}"
                        prob += decision_variables["transfer_in_bench_free"][i] == 0, f"NonBenchFree{i}"
                        prob += decision_variables["stay_bench"][i] == 0, f"NonBenchStay{i}"
                        prob += decision_variables["bench_to_starting"][i] == 0, f"NonBenchSwap{i}"
                        
            
        for i in df.index:
            prob += (decision_variables["transfer_in_starting_free"][i] + 
                    decision_variables["transfer_in_starting_paid"][i] + 
                    decision_variables["stay_starting"][i] + 
                    decision_variables["bench_to_starting"][i] + 
                    decision_variables["transfer_in_bench_free"][i] +
                    decision_variables["transfer_in_bench_paid"][i] +
                    decision_variables["stay_bench"][i] +
                    decision_variables["starting_to_bench"][i] +
                    decision_variables["transfer_out_starting_free"][i] +
                    decision_variables["transfer_out_starting_paid"][i] +
                    decision_variables["transfer_out_bench_free"][i] +
                    decision_variables["transfer_out_bench_paid"][i]) <= 1, f"Exclusive_{i}"
            
        filled_players = df[df['filled'] == True].index
        for i in filled_players:
            prob += decision_variables["transfer_in_bench_free"][i] == 0, f"Filled__starting{i}"
            prob += decision_variables["transfer_in_bench_paid"][i] == 0, f"Filled__bench{i}"
                

        if gw > 1:
            prob += lpSum(decision_variables["transfer_out_starting_free"][i] for i in df.index) == \
                    lpSum(decision_variables["transfer_in_starting_free"][i] for i in df.index), "StartingFreeTransfersBalance"

            prob += lpSum(decision_variables["transfer_out_starting_paid"][i] for i in df.index) == \
                    lpSum(decision_variables["transfer_in_starting_paid"][i] for i in df.index), "StartingPaidTransfersBalance"

            prob += lpSum(decision_variables["transfer_out_bench_free"][i] for i in df.index) == \
                    lpSum(decision_variables["transfer_in_bench_free"][i] for i in df.index), "BenchFreeTransfersBalance"

            prob += lpSum(decision_variables["transfer_out_bench_paid"][i] for i in df.index) == \
                    lpSum(decision_variables["transfer_in_bench_paid"][i] for i in df.index), "BenchPaidTransfersBalance"
            
            prob += lpSum(decision_variables["bench_to_starting"][i] for i in df.index) == \
                    lpSum(decision_variables["starting_to_bench"][i] for i in df.index), "BenchToStartingTransfersBalance"
            
            prob += lpSum([decision_variables["transfer_in_starting_free"][i] + decision_variables["transfer_in_bench_free"][i] for i in df.index]) <= transfer_limit, "TotalTransferInFreeLimit"
            prob += lpSum([decision_variables["transfer_out_starting_free"][i] + decision_variables["transfer_out_bench_free"][i] for i in df.index]) <= transfer_limit, "TotalTransferOutFreeLimit"

            previous_starters = previous_squad[previous_squad['role'] == 'starting']['player_id'] # Get the player IDs of the previous starters
            previous_bench = previous_squad[previous_squad['role'] == 'bench']['player_id']
            starter_indexes = df[df['player_id'].isin(previous_starters)].index # Get the indexes of the previous starters
            bench_indexes = df[df['player_id'].isin(previous_bench)].index # Get the indexes of the previous bench players
            
            for index in df.index:
                if index not in starter_indexes:
                    prob += decision_variables["transfer_out_starting_free"][index] == 0, f"Constraint_transfer_out_starting_free_{index}"
                    prob += decision_variables["transfer_out_starting_paid"][index] == 0, f"Constraint_transfer_out_starting_paid_{index}"
                    prob += decision_variables["starting_to_bench"][index] == 0, f"Constraint_starting_to_bench_{index}"
                    prob += decision_variables["stay_starting"][index] == 0, f"Constraint_stay_starting_{index}"
            for index in df.index:
                if index not in bench_indexes:
                    prob += decision_variables["transfer_out_bench_free"][index] == 0, f"Constraint_transfer_out_bench_free_{index}"
                    prob += decision_variables["transfer_out_bench_paid"][index] == 0, f"Constraint_transfer_out_bench_paid_{index}"
                    prob += decision_variables["bench_to_starting"][index] == 0, f"Constraint_bench_to_starting_{index}"
                    prob += decision_variables["stay_bench"][index] == 0, f"Constraint_stay_bench_{index}"





## Defining the problem ##


In [133]:
class Transfer_Squad_Selection:
    def Problem(self, df, season, gw, previous_squad, new_budget_limit, transfer_limit):
        if gw == 1:
            budget_limit = [1000]
        else:   
            budget_limit = [new_budget_limit]   

        prob = LpProblem("TransferSquadSelection", LpMaximize)

        decision_variables = constraints().get_decision_variables(df)

        constraints().set_objective_function(df, decision_variables, prob)
        constraints().apply_formation_constraints(df, decision_variables, prob)
        constraints().apply_transfer_constraints(df, decision_variables, prob, budget_limit[-1])
        constraints().apply_player_constraints(df, decision_variables, prob, previous_squad, gw, transfer_limit, season)

        return (
            prob,
            {var_name: decision_variables[var_name] for var_name in decision_variables}
        )

## Attempt to get dataset ##

In [134]:
season = '2020-21'
df = get_dataset().run(season)

## Execute ##

In [135]:
squads = []
budgets = []
all_tables = []
budgets = [1000]
total_score = 0
total_xp = 0
total_paid_transfers = 0
transfer_limit = 1

bench_boost_activated = False
triple_captain_activated = False
free_hit_activated = False
wildcard_activated = False
average_xp_bench = []
average_xp_captain = []
average_paid_transfers = []
average_xp_squad = []


for gw in range(1, 39):
    print("----------------------------------")
    df_gw = df[df['GW'] == gw]
    next_gw_df = df[df['GW'] == gw + 1]

    if gw == 1:
        (
            prob,
            decision_variables
        ) = Transfer_Squad_Selection().Problem(
            df_gw, 
            season, 
            gw, 
            None, 
            None, 
            None
        )
                   

    else:
        (
            prob,
            decision_variables
        ) = Transfer_Squad_Selection().Problem(
            df_gw, 
            season, 
            gw, 
            squads[-1], 
            budgets[-1], 
            transfer_limit
        )
                                    
                                    
    prob.solve()

    transfer_in_starting_free = decision_variables["transfer_in_starting_free"]
    transfer_in_starting_paid = decision_variables["transfer_in_starting_paid"]
    stay_starting = decision_variables["stay_starting"]
    bench_to_starting = decision_variables["bench_to_starting"]
    transfer_in_bench_free = decision_variables["transfer_in_bench_free"]
    transfer_in_bench_paid = decision_variables["transfer_in_bench_paid"]
    stay_bench = decision_variables["stay_bench"]
    starting_to_bench = decision_variables["starting_to_bench"]
    transfer_out_starting_free = decision_variables["transfer_out_starting_free"]
    transfer_out_starting_paid = decision_variables["transfer_out_starting_paid"]
    transfer_out_bench_free = decision_variables["transfer_out_bench_free"]
    transfer_out_bench_paid = decision_variables["transfer_out_bench_paid"]
  

    starters_free = [i for i in df_gw.index if transfer_in_starting_free[i].varValue == 1]
    starters_paid = [i for i in df_gw.index if transfer_in_starting_paid[i].varValue == 1]
    starters_from_bench = [i for i in df_gw.index if bench_to_starting[i].varValue == 1]
    starters_stayed = [i for i in df_gw.index if stay_starting[i].varValue == 1]
    starters = starters_free + starters_paid + starters_from_bench + starters_stayed

    bench_free = [i for i in df_gw.index if transfer_in_bench_free[i].varValue == 1]
    bench_paid = [i for i in df_gw.index if transfer_in_bench_paid[i].varValue == 1]
    bench_from_starters = [i for i in df_gw.index if starting_to_bench[i].varValue == 1]
    bench_stayed = [i for i in df_gw.index if stay_bench[i].varValue == 1]
    bench = bench_free + bench_paid + bench_from_starters + bench_stayed

    full_squad = df_gw.loc[starters + bench]

    functions().assign_captaincies(full_squad, starters)
    sorted_squad = functions().sort_squad(full_squad)
    squads.append(sorted_squad)
    total_xp += full_squad[full_squad['role'] == 'starting']['xP'].sum()


    if gw > 1:
        transfer_limit -= len(starters_free) + len(bench_free) - 1
        transfer_limit = min(transfer_limit, 5)   
        total_paid_transfers += len(starters_paid) + len(bench_paid)
    
    table = []
    lists = [
        'stay_starting', 'stay_bench', 'transfer_in_starting_free', 'transfer_out_starting_free', 
        'transfer_in_starting_paid',  'transfer_out_starting_paid', 'bench_to_starting',
        'transfer_in_bench_free', 'transfer_out_bench_free', 'starting_to_bench'
    ]
    for list_name in lists:
        player_list = eval(list_name)
        players = [df_gw.loc[i, 'name'] for i in df_gw.index if player_list[i].varValue == 1]
        table.append([list_name, ", ".join(players)])
    all_tables.append(table)

    total_score, data = functions().data_calculations(full_squad, budgets[-1], total_score, len(starters_paid))
    new_budget_limit = budgets[-1] - full_squad['value'].sum() + next_gw_df[next_gw_df['player_id'].isin(full_squad['player_id'].unique())]['value'].sum()
    budgets.append(new_budget_limit)

    bench_condition = WildCards().bench_boost(full_squad, average_xp_bench, bench_boost_activated)
    if bench_condition:
        total_score += functions().update_data(data, full_squad[full_squad['role'] == 'bench']['total_points'].sum())
        bench_boost_activated = True
        bench_gw = gw

    captain_condition = WildCards().triple_captain(full_squad, average_xp_captain, triple_captain_activated)
    if captain_condition:   
        total_score += functions().update_data(data, full_squad[full_squad['captain'] == 1]['total_points'].sum())
        triple_captain_activated = True
        captain_gw = gw

    wildcard_condition = WildCards().wildcard(len(starters_paid) + len(bench_paid), average_paid_transfers, wildcard_activated)
    if wildcard_condition:
        total_score += functions().update_data(data, (len(starters_paid) + len(bench_paid) - 1) * 4)
        wildcard_activated = True
        wildcard_gw = gw

    freehit_condition = WildCards().freehit(full_squad, average_xp_squad, free_hit_activated)
    if freehit_condition:
        total_score += functions().update_data(data, (len(starters_paid) + len(bench_paid) - 1) * 4)
        squads.pop() 
        free_hit_activated = True
        freehit_gw = gw

    print(f"Gameweek {gw}") 
    if pulp.LpStatus[prob.status] == 'Optimal':
        print("All constraints were met. Solution found.")
    else:
        print(f"Solution not found. Status: {pulp.LpStatus[prob.status]}")
    print(tabulate(sorted_squad, headers='keys', tablefmt='fancy_grid'))
    print(tabulate(data, headers=["Category", "Value"], tablefmt="fancy_grid"))
    print(tabulate(table, headers=["Type of Transfer", "Players"], tablefmt="fancy_grid"))

data = [
    ["Total Season Points", total_score],
    ["Total Paid Transfers", total_paid_transfers],
    ["Total Expected Points", total_xp],
    ["Total Points Without Transfer Penalties", total_score + (total_paid_transfers) * 4],
    ["Wildcard Gameweek", wildcard_gw],
    ["Bench Boost Gameweek", bench_gw],
    ["Triple Captain Gameweek", captain_gw],
    ["Free Hit Gameweek", freehit_gw]
]
print(tabulate(data, headers=["Category", "Value"], tablefmt="fancy_grid"))



----------------------------------
Gameweek 1
All constraints were met. Solution found.
╒═══════╤═════════════════════════╤════════════╤════════════════╤══════════╤═══════════╤══════╤════════════════╤═════════╤═══════════╤════════════════╤═════════════╕
│       │ name                    │ position   │ team           │ role     │   minutes │   xP │   total_points │   value │   captain │   vice_captain │   player_id │
╞═══════╪═════════════════════════╪════════════╪════════════════╪══════════╪═══════════╪══════╪════════════════╪═════════╪═══════════╪════════════════╪═════════════╡
│ 26296 │ Vicente Guaita          │ GK         │ Crystal Palace │ starting │        90 │  2.5 │             10 │      50 │         0 │              0 │         692 │
├───────┼─────────────────────────┼────────────┼────────────────┼──────────┼───────────┼──────┼────────────────┼─────────┼───────────┼────────────────┼─────────────┤
│  8854 │ Gabriel Magalhães       │ DEF        │ Arsenal        │ starting │      